<a href="https://colab.research.google.com/github/saraduquej/Analitica-de-negocios/blob/main/Reto1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

#**Caso de estudio**: detección de Riesgo SARLAFT en una Entidad Financiera


En este documento se desarrolla y analiza un modelo de clasificación Naive Bayes con el propósito de fortalecer el sistema de prevención de lavado de activos y financiación del terrorismo (SARLAFT) en una entidad financiera. La motivación surge de la necesidad de mejorar la identificación temprana de clientes con alto riesgo, ya que los controles tradicionales no siempre logran detectar oportunamente operaciones sospechosas. Para ello, el modelo se apoya en el análisis del comportamiento financiero de los clientes y en la detección de cambios inusuales en su patrimonio, permitiendo clasificar el nivel de riesgo.

De acuerdo con lo anterior, las variables que se analizan son:

**Edad:** número de años de vida del cliente al momento de la evaluación.

**Ingresos (USD):** representa el dinero que recibe el cliente como resultado de una actividad comercial, laboral o económica, medido en dólares estadounidenses.

**Gastos (USD):** hace referencia a los egresos mensuales del cliente, expresados en dólares estadounidenses. Incluye pagos asociados a consumo, obligaciones financieras y otros compromisos económicos.

**Numero de tarjetas de crédito:** cantidad de tarjetas de crédito que posee el cliente al momento de la evaluación.

**Monto transado tarjetas (USD):** indica la cantidad total de dólares estadounidenses que el cliente mueve mensualmente a través de sus tarjetas del banco.

**Porcentaje crecimiento del patrimonio:** mide el crecimiento del patrimonio del cliente expresado en términos porcentuales durante un periodo determinado. El patrimonio se entiende como el conjunto total de bienes, tanto materiales como inmateriales, que son valorables económicamente.


**0. Carga de librerias de trabajo**

In [ ]:
import numpy as np
import pandas as pd

# Librerias especificas
from sklearn.naive_bayes import GaussianNB
from sklearn.metrics import confusion_matrix

from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


**1. Se cargan los datos de trabajo de la base de datos de lavado de activos**

In [ ]:
nxl = '/content/drive/MyDrive/Analítica de negocios/2. LavadoActivos.xlsx'
XDB = pd.read_excel(nxl,sheet_name=0)
XDB.head(100)

# Seleccionamos las variables de trabajo
XD = XDB [['edad', 'ingresos_usd','gastos_usd', 'num_tarjetas_credito', 'monto_transado_tarjetas_usd', 'porcentaje_crecimiento_patrimonio']]
XD.head(10)
yd = XDB[['lavado_activos']]
yd.head()

,lavado_activos
0,1
1,1
2,0
3,0
4,0


**2. Se implementa el Modelo Naive Bayes**

In [ ]:
np.set_printoptions(suppress=True,precision=3)
mnb = GaussianNB()
mnb.fit(XD,yd)

# Mostramos las medias de las variables
u = mnb.theta_
sigma = mnb.var_; sigma = np.sqrt(sigma)
print("'edad', 'ingresos_usd', 'gastos_usd', 'num_tarjetas_credito', 'monto_transado_tarjetas_usd', 'porcentaje_crecimiento_patrimonio'")
print(u)
print()
print("Las desviaciones son:")
print(sigma)
print()
print('los limites superiores de las variables son:',u+sigma)
print()
print('los limites inferiores de las variables son:',u-sigma)
print()

'edad', 'ingresos_usd', 'gastos_usd', 'num_tarjetas_credito', 'monto_transado_tarjetas_usd', 'porcentaje_crecimiento_patrimonio'
[[   45.47   6690.733  3601.75      1.955  3090.464    44.687]
 [   44.3   70262.304 38707.284     2.016 33835.62     44.821]]

Las desviaciones son:
[[    14.711   6206.964   3219.016      2.535   2456.859     14.173]
 [    15.052 113412.265  65902.557      2.557  62991.391     15.479]]

los limites superiores de las variables son: [[    60.18   12897.697   6820.765      4.49    5547.323     58.86 ]
 [    59.352 183674.569 104609.842      4.574  96827.011     60.299]]

los limites inferiores de las variables son: [[    30.759    483.768    382.734     -0.581    633.605     30.514]
 [    29.248 -43149.96  -27195.273     -0.541 -29155.771     29.342]]



/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:1408: DataConversionWarning: A column-vector y was passed when a 1d array was expected. Please change the shape of y to (n_samples, ), for example using ravel().
  y = column_or_1d(y, warn=True)


**Análisis de resultados**

Al analizar las medias, desviaciones y límites, se observa que la edad presenta valores muy similares entre los clientes sin riesgo (0) y con riesgo de lavado de activos (1), con rangos similares entre ambos grupos, por lo que no es una variable diferenciadora fuerte.

En contraste, los ingresos y gastos muestran diferencias claras, pues el grupo con riesgo (1) presenta promedios mucho más altos y desviaciones mayores, lo que amplía significativamente los límites superiores, y genera límites inferiores negativos. Esto evidencia una alta variabilidad e inestabilidad en los flujos financieros de los clientes riesgosos, una señal relevante dentro del análisis SARLAFT.

Por otro lado, el número de tarjetas de crédito es ligeramente mayor en el grupo con riesgo (1), con desviaciones mínimas.
Además, el monto transado en tarjetas es una de las variables que mejor diferencia a ambos grupos, ya que los clientes con riesgo (1) presentan volúmenes de transacción mucho más altos y rangos más amplios, reflejando grandes y variables movimientos de dinero.

Finalmente, el porcentaje de crecimiento del patrimonio muestra valores muy similares entre los clientes sin riesgo (0) y con riesgo de lavado de activos (1), con medias similares y desviaciones estándar cercanas, lo que sugiere que el crecimiento patrimonial no es un factor diferenciador fuerte por sí solo.


**3. Se analiza el modelo usando la matriz de confusión**

In [ ]:
ydp = mnb.predict(XD)
cm = confusion_matrix(yd,ydp)
print(cm)

# Métricas de la matriz de confusión
VN = cm[0,0]
FP = cm[0,1]
FN = cm[1,0]
VP = cm[1,1]
TDatos = len(XDB)

# 1. Exactitud
Ex = (VP+VN)/TDatos
print("Exactitud: ",Ex)

# 2. Tasa error
TEr = (FP+FN)/TDatos
print("Tasa error: ",TEr)

# 3. Sensibilidad
Se = VP/(VP+FN)
print("Sensibilidad: ",Se)

# 4. Especificidad
Es = VN/(VN+FP)
print("Especificidad: ",Es)

# 5. Precisión
Pr = VP/(VP+FP)
print("Precisión: ",Pr)

# 6. Predicción negativa
PrN = VN/(VN+FN)
print("Predicción negativa: ",PrN)

[[2179   44]
 [ 145  782]]
Exactitud:  0.94
Tasa error:  0.06
Sensibilidad:  0.8435814455231931
Especificidad:  0.9802069275753487
Precisión:  0.9467312348668281
Predicción negativa:  0.9376075731497419


**Análisis de resultados**

El modelo tiene un buen desempeño, reflejado en una exactitud del 94% y una tasa de error del 6%, lo que indica que la gran mayoría de las decisiones son correctas. La especificidad del 98% evidencia que clasifica de manera correcta a los clientes sin riesgo, mientras que la precisión del 95% señala que, cuando el modelo genera una alerta, esta suele ser acertada, evitando falsos positivos innecesarios.


La sensibilidad (84%) indica que el modelo detecta la mayoría de los clientes con riesgo de incurrir en SARLAFT, aunque se le escapan algunos. La predicción negativa (94%) refuerza que cuando el modelo dice que un cliente no es riesgoso, esa decisión suele ser confiable. En conjunto, es un modelo sólido para fortalecer el SARLAFT.

**4. Se evalúa un solicitante**

In [ ]:
XP = [58, 31692, 19139, 2, 10493, 39.61] # edad, ingresos_usd, gastos_usd, num_tarjetas_credito, monto_transado_tarjetas_usd, porcentaje_crecimiento_patrimonio
ydc = mnb.predict([XP])
print(ydc)

if ydc==1:
  print("El cliente tiene alto riesgo de incurrir en lavados de activos")
else:
  print("El cliente no tiene un riesgo alto de incurrir en lavado de activos")

[1]
El cliente tiene alto riesgo de incurrir en lavados de activos


/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but GaussianNB was fitted with feature names
  warnings.warn(


**Análisis de resultados**

En esta etapa se evalúa un solicitante utilizando el modelo Naive Bayes, tomando como entrada un conjunto de datos que proviene de la base de datos de la entidad financiera. Estas variables representan el perfil socioeconómico y transaccional del cliente (edad, ingresos, gastos, número de tarjetas, monto transado y crecimiento del patrimonio), y son analizadas para estimar su nivel de riesgo dentro del sistema SARLAFT.
Con base en la información registrada, el modelo clasifica al solicitante como cliente con alto riesgo de incurrir en lavado de activos.